In [31]:
# ResNet-50 + ViT Cross-Attention Fusion for Thoracic Disease Classification
# Model 3: ResNet-50 + ViT with Cross-Attention Fusion

import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import cv2
import os
from tensorflow_addons.losses import SigmoidFocalCrossEntropy
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.10.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [32]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset configuration
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4

# Transfer learning strategy 
USE_TRANSFER_LEARNING = True
FREEZE_EPOCHS = 12  # Train ViT/Cross-attention first, then fine-tune everything

# Model configuration
CNN_FEATURE_DIM = 2048  # ResNet-50 output dimension
VIT_PATCH_SIZE = 16
VIT_EMBED_DIM = 256
VIT_NUM_HEADS = 8
VIT_NUM_LAYERS = 6
CROSS_ATTENTION_DIM = 512

# Class names (including No_Finding as you suggested)
CLASS_NAMES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
    "Nodule", "No Finding", "Pleural_Thickening", "Pneumonia", "Pneumothorax"
]
NUM_CLASSES = len(CLASS_NAMES)

In [33]:
# =============================================================================
# CUSTOM LAYERS AND COMPONENTS
# =============================================================================

class PatchExtractor(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim
        })
        return config

class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        if embed_dim % num_heads != 0:
            raise ValueError(f"embed_dim ({embed_dim}) should be divisible by num_heads ({num_heads})")
        self.projection_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads
        })
        return config

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout_rate = dropout
        
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.mlp = keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        mlp_output = self.mlp(out1)
        mlp_output = self.dropout2(mlp_output, training=training)
        return self.layernorm2(out1 + mlp_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "mlp_dim": self.mlp_dim,
            "dropout": self.dropout_rate
        })
        return config

class CrossAttentionFusion(layers.Layer):
    def __init__(self, attention_dim, num_heads=8, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention_dim = attention_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout
        
        # Project CNN and ViT features to common dimension
        self.cnn_projection = layers.Dense(attention_dim)
        self.vit_projection = layers.Dense(attention_dim)
        
        # Cross-attention layer
        self.cross_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=attention_dim // num_heads,
            dropout=dropout
        )
        
        # Layer normalization and dropout
        self.layernorm = layers.LayerNormalization(epsilon=1e-6)
        self.dropout = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        cnn_features, vit_features = inputs
        
        # Project features to common dimension
        cnn_proj = self.cnn_projection(cnn_features)
        vit_proj = self.vit_projection(vit_features)
        
        # Cross-attention: CNN features as queries, ViT features as keys/values
        attended_features = self.cross_attention(
            query=cnn_proj,
            key=vit_proj,
            value=vit_proj,
            training=training
        )
        
        # Residual connection and normalization
        output = self.layernorm(cnn_proj + self.dropout(attended_features, training=training))
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "attention_dim": self.attention_dim,
            "num_heads": self.num_heads,
            "dropout": self.dropout_rate
        })
        return config

In [34]:
# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================

def create_resnet_vit_model(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    num_classes=NUM_CLASSES,
    patch_size=VIT_PATCH_SIZE,
    vit_embed_dim=VIT_EMBED_DIM,
    vit_num_heads=VIT_NUM_HEADS,
    vit_num_layers=VIT_NUM_LAYERS,
    cross_attention_dim=CROSS_ATTENTION_DIM
):
    """
    Create ResNet-50 + ViT with Cross-Attention Fusion model
    """
    inputs = layers.Input(shape=input_shape)
    
    # ===== ResNet-50 CNN Branch =====
    resnet_base = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=inputs,
        pooling='avg'
    )
    
    # Start with frozen ResNet for transfer learning
    resnet_base.trainable = False  # Will be unfrozen later
    
    # Extract CNN features
    cnn_features = resnet_base.output  # Shape: (batch_size, 2048)
    
    # ===== Vision Transformer Branch =====
    num_patches = (IMAGE_SIZE // patch_size) ** 2
    
    # Extract patches
    patches = PatchExtractor(patch_size)(inputs)
    
    # Encode patches
    encoded_patches = PatchEncoder(num_patches, vit_embed_dim)(patches)
    
    # Transformer blocks
    for i in range(vit_num_layers):
        encoded_patches = TransformerBlock(
            embed_dim=vit_embed_dim,
            num_heads=vit_num_heads,
            mlp_dim=vit_embed_dim * 4,
            dropout=0.1
        )(encoded_patches)
    
    # Global average pooling for ViT features
    vit_features = layers.GlobalAveragePooling1D()(encoded_patches)  # Shape: (batch_size, vit_embed_dim)
    
    # ===== Cross-Attention Fusion =====
    # Expand dimensions for cross-attention (add sequence dimension)
    cnn_features_expanded = tf.expand_dims(cnn_features, axis=1)  # (batch_size, 1, 2048)
    vit_features_expanded = tf.expand_dims(vit_features, axis=1)  # (batch_size, 1, vit_embed_dim)
    
    # Apply cross-attention fusion
    fused_features = CrossAttentionFusion(
        attention_dim=cross_attention_dim,
        num_heads=8,
        dropout=0.1
    )([cnn_features_expanded, vit_features_expanded])
    
    # Remove sequence dimension
    fused_features = tf.squeeze(fused_features, axis=1)  # (batch_size, cross_attention_dim)
    
    # ===== Classification Head =====
    x = layers.Dropout(0.3)(fused_features)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    # Multi-label classification output
    outputs = layers.Dense(num_classes, activation='sigmoid', name='predictions')(x)
    
    model = keras.Model(inputs, outputs, name='ResNet50_ViT_CrossAttention')
    return model

In [35]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================

def preprocess_image(image_path, target_size=(IMAGE_SIZE, IMAGE_SIZE)):
    """Preprocess a single image for training"""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not load image: {image_path}")
    
    # Resize image
    image = cv2.resize(image, target_size)
    
    # Convert to RGB (repeat grayscale channel)
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    
    # Normalize to [0, 1]
    image = image.astype(np.float32) / 255.0
    
    return image

def create_data_generator(df, batch_size, is_training=True):
    """Create a data generator for training/validation"""
    def generator():
        while True:
            if is_training:
                df_batch = df.sample(n=batch_size).reset_index(drop=True)
            else:
                for i in range(0, len(df), batch_size):
                    df_batch = df.iloc[i:i+batch_size].reset_index(drop=True)
                    
                    if len(df_batch) == 0:
                        break
                    
                    images = []
                    labels = []
                    
                    for idx, row in df_batch.iterrows():
                        try:
                            image = preprocess_image(row['image_path'])
                            images.append(image)
                            
                            # Multi-label encoding
                            label = [row[class_name] for class_name in CLASS_NAMES]
                            labels.append(label)
                            
                        except Exception as e:
                            print(f"Error loading image {row['image_path']}: {e}")
                            continue
                    
                    if len(images) > 0:
                        yield np.array(images), np.array(labels)
                        
                if not is_training:
                    break
            
            if is_training:
                images = []
                labels = []
                
                for idx, row in df_batch.iterrows():
                    try:
                        image = preprocess_image(row['image_path'])
                        images.append(image)
                        
                        # Multi-label encoding
                        label = [row[class_name] for class_name in CLASS_NAMES]
                        labels.append(label)
                        
                    except Exception as e:
                        print(f"Error loading image {row['image_path']}: {e}")
                        continue
                
                if len(images) > 0:
                    yield np.array(images), np.array(labels)
    
    return generator

In [36]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================

def preprocess_image(image_path, target_size=(IMAGE_SIZE, IMAGE_SIZE)):
    """Preprocess a single image for training"""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not load image: {image_path}")
    
    # Resize image
    image = cv2.resize(image, target_size)
    
    # Convert to RGB (repeat grayscale channel)
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    
    # Normalize to [0, 1]
    image = image.astype(np.float32) / 255.0
    
    return image

def create_data_generator(df, batch_size, is_training=True):
    """Create a data generator for training/validation"""
    def generator():
        while True:
            if is_training:
                df_batch = df.sample(n=batch_size).reset_index(drop=True)
            else:
                for i in range(0, len(df), batch_size):
                    df_batch = df.iloc[i:i+batch_size].reset_index(drop=True)
                    
                    if len(df_batch) == 0:
                        break
                    
                    images = []
                    labels = []
                    
                    for idx, row in df_batch.iterrows():
                        try:
                            image = preprocess_image(row['image_path'])
                            images.append(image)
                            
                            # Multi-label encoding
                            label = [row[class_name] for class_name in CLASS_NAMES]
                            labels.append(label)
                            
                        except Exception as e:
                            print(f"Error loading image {row['image_path']}: {e}")
                            continue
                    
                    if len(images) > 0:
                        yield np.array(images), np.array(labels)
                        
                if not is_training:
                    break
            
            if is_training:
                images = []
                labels = []
                
                for idx, row in df_batch.iterrows():
                    try:
                        image = preprocess_image(row['image_path'])
                        images.append(image)
                        
                        # Multi-label encoding
                        label = [row[class_name] for class_name in CLASS_NAMES]
                        labels.append(label)
                        
                    except Exception as e:
                        print(f"Error loading image {row['image_path']}: {e}")
                        continue
                
                if len(images) > 0:
                    yield np.array(images), np.array(labels)
    
    return generator

In [37]:
# =============================================================================
# TRAINING SETUP
# =============================================================================

# Split dataset
print("Splitting dataset...")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=None)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Create model
print("Creating ResNet-50 + ViT model...")
model = create_resnet_vit_model()

# Print model summary
model.summary()

# Compile model
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

# Use binary crossentropy for multi-label classification
# You can also experiment with focal loss
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',  # or SigmoidFocalCrossEntropy()
    metrics=['binary_accuracy', 'AUC']
)

Splitting dataset...
Training samples: 41105
Validation samples: 10277
Creating ResNet-50 + ViT model...
Model: "ResNet50_ViT_CrossAttention"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_4[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                 

In [38]:
# =============================================================================
# CALLBACKS
# =============================================================================

# Model checkpoint
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./output/resnet50_vit_best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# Callbacks
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./output/resnet50_vit_best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.001
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

# Transfer learning callback - unfreeze ResNet after initial training
class UnfreezeResNetCallback(tf.keras.callbacks.Callback):
    def __init__(self, unfreeze_epoch=FREEZE_EPOCHS):
        super().__init__()
        self.unfreeze_epoch = unfreeze_epoch
        self.unfrozen = False
    
    def on_epoch_begin(self, epoch, logs=None):
        if epoch == self.unfreeze_epoch and not self.unfrozen:
            print(f"\n{'='*60}")
            print(f"UNFREEZING ResNet-50 at epoch {epoch + 1}")
            print(f"{'='*60}")
            
            # Find and unfreeze ResNet layers
            for layer in self.model.layers:
                if 'resnet50' in layer.name.lower():
                    layer.trainable = True
                    print(f"✓ Unfrozen: {layer.name}")
                    break
            
            # Reduce learning rate for fine-tuning (important!)
            old_lr = float(self.model.optimizer.learning_rate)
            new_lr = old_lr * 0.1
            self.model.optimizer.learning_rate = new_lr
            
            print(f"📉 Reduced learning rate: {old_lr:.2e} → {new_lr:.2e}")
            print(f"🎯 Now fine-tuning entire hybrid model")
            print(f"{'='*60}\n")
            
            self.unfrozen = True
            
            # Recompile to ensure changes take effect
            self.model.compile(
                optimizer=self.model.optimizer,
                loss=self.model.loss,
                metrics=self.model.metrics
            )

unfreeze_callback = UnfreezeResNetCallback(FREEZE_EPOCHS)

In [ ]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

# Create data generators
print("Creating data generators...")
train_generator = tf.data.Dataset.from_generator(
    create_data_generator(train_df, BATCH_SIZE, is_training=True),
    output_signature=(
        tf.TensorSpec(shape=(None, IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, NUM_CLASSES), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

val_generator = tf.data.Dataset.from_generator(
    create_data_generator(val_df, BATCH_SIZE, is_training=False),
    output_signature=(
        tf.TensorSpec(shape=(None, IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, NUM_CLASSES), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

# Calculate steps per epoch
steps_per_epoch = len(train_df) // BATCH_SIZE
validation_steps = len(val_df) // BATCH_SIZE

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Validation steps: {validation_steps}")

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint_callback, early_stopping, lr_scheduler, unfreeze_callback],
    verbose=1
)

# Save final model
model.save('./output/resnet50_vit_final_model.keras')
print("Training completed!")

Creating data generators...
Steps per epoch: 2569
Validation steps: 642
Epoch 1/30
2569/2569 [==============================] - ETA: 0s - loss: 0.3719 - binary_accuracy: 0.8820 - auc: 0.5188
Epoch 1: val_loss improved from inf to 0.39058, saving model to ./output\resnet50_vit_best_model.keras
2569/2569 [==============================] - 1264s 489ms/step - loss: 0.3719 - binary_accuracy: 0.8820 - auc: 0.5188 - val_loss: 0.3906 - val_binary_accuracy: 0.9007 - val_auc: 0.5964 - lr: 1.0000e-04
Epoch 2/30
2569/2569 [==============================] - ETA: 0s - loss: 0.3322 - binary_accuracy: 0.9005 - auc: 0.5609

In [ ]:
# =============================================================================
# EVALUATION AND VISUALIZATION
# =============================================================================

# Plot training history
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['binary_accuracy'], label='Training Accuracy')
plt.plot(history.history['val_binary_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['auc'], label='Training AUC')
plt.plot(history.history['val_auc'], label='Validation AUC')
plt.title('Model AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate on validation set
print("Evaluating on validation set...")

# Collect predictions and true labels
val_predictions = []
val_true_labels = []

print("Collecting validation predictions...")
for batch_num, (images, labels) in enumerate(val_generator.take(validation_steps)):
    batch_predictions = model.predict(images, verbose=0)
    val_predictions.extend(batch_predictions)
    val_true_labels.extend(labels.numpy())
    
    if (batch_num + 1) % 10 == 0:
        print(f"Processed {batch_num + 1}/{validation_steps} validation batches")

# Convert to numpy arrays
val_true = np.array(val_true_labels)
val_pred = np.array(val_predictions)

print(f"Validation set shape: {val_true.shape}")

# Calculate metrics for each class
print("\nPer-class AUC scores:")
auc_scores = []
for i, class_name in enumerate(CLASS_NAMES):
    try:
        if len(np.unique(val_true[:, i])) > 1:  # Check if both classes are present
            auc = roc_auc_score(val_true[:, i], val_pred[:, i])
            auc_scores.append(auc)
            print(f"{class_name}: {auc:.4f}")
        else:
            auc_scores.append(0.0)
            print(f"{class_name}: No positive samples in validation set")
    except ValueError as e:
        auc_scores.append(0.0)
        print(f"{class_name}: Error calculating AUC - {e}")

# Calculate mean AUC
mean_auc = np.mean([score for score in auc_scores if score > 0])
print(f"\nMean AUC (excluding classes with no positives): {mean_auc:.4f}")

# Print classification report (for binary predictions at threshold 0.5)
val_pred_binary = (val_pred > 0.5).astype(int)
print("\nClassification Report (threshold = 0.5):")
print(classification_report(
    val_true, 
    val_pred_binary, 
    target_names=CLASS_NAMES, 
    zero_division=0
))

# Plot class distribution in validation predictions
plt.figure(figsize=(15, 6))
val_pred_counts = np.sum(val_pred_binary, axis=0)
val_true_counts = np.sum(val_true, axis=0)

x = np.arange(len(CLASS_NAMES))
width = 0.35

plt.bar(x - width/2, val_true_counts, width, label='True', alpha=0.8)
plt.bar(x + width/2, val_pred_counts, width, label='Predicted', alpha=0.8)

plt.xlabel('Classes')
plt.ylabel('Count')
plt.title('True vs Predicted Class Distribution (Validation Set)')
plt.xticks(x, CLASS_NAMES, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

# Create confusion matrix for top predicted class
print("\nTop prediction accuracy:")
val_true_top = np.argmax(val_true, axis=1)
val_pred_top = np.argmax(val_pred, axis=1)
top_accuracy = np.mean(val_true_top == val_pred_top)
print(f"Top prediction accuracy: {top_accuracy:.4f}")

print(f"\nModel training and evaluation completed!")
print(f"Best model saved at: ./output/resnet50_vit_best_model.keras")
print(f"Final model saved at: ./output/resnet50_vit_final_model.keras")

print("\nModel architecture created successfully!")
print("To train the model:")
print("1. Load your dataset CSV file")
print("2. Uncomment the training section")
print("3. Run the notebook")
print(f"\nModel expects images of size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")